In [ ]:
#ADD THESE PATHS AND RUN

MODEL_PATH = '/kaggle/working/mouse_axis_finetuned.pt'
IMAGE_DIR = '/kaggle/input/datasets/your_dataset/images'

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF
import timm
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CLASSES = ['coronal', 'sagittal', 'axial']
NUM_CLASSES = len(CLASSES)
MODEL_NAME = 'convnext_tiny'
IMG_SIZE = 320
BATCH_SIZE = 16

Image.MAX_IMAGE_PIXELS = None
VALID_EXTS = ('.tif', '.tiff', '.png', '.jpg', '.jpeg', '.jfif', '.bmp')


In [ ]:
def normalize_contrast(pil_img, low_pct=1.0, high_pct=99.0, use_clahe=False):
    gray = np.array(pil_img.convert('L')).astype(np.float32)
    lo, hi = np.percentile(gray, [low_pct, high_pct])
    if hi - lo < 1e-3:
        hi = lo + 1e-3
    gray = np.clip((gray - lo) / (hi - lo), 0.0, 1.0)
    if use_clahe:
        try:
            from skimage.exposure import equalize_adapthist
            gray = equalize_adapthist(gray, clip_limit=0.01)
        except ImportError:
            pass
    gray_u8 = (gray * 255).astype(np.uint8)
    return Image.fromarray(gray_u8).convert('RGB')


def is_image_file(path):
    return os.path.splitext(path)[1].lower() in VALID_EXTS


def load_image(path):
    import tifffile
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.tif', '.tiff'):
        arr = tifffile.imread(path)
        if arr.ndim == 2:
            arr = np.stack([arr] * 3, axis=-1)
        elif arr.ndim == 3 and arr.shape[0] in (1, 3, 4):
            arr = np.transpose(arr, (1, 2, 0))
        if arr.shape[-1] == 4:
            arr = arr[..., :3]
        arr = arr.astype(np.float32)
        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8) * 255
        img = Image.fromarray(arr.astype(np.uint8)).convert('RGB')
    else:
        raw = Image.open(path)
        if raw.mode == 'RGBA':
            bg = Image.new('RGB', raw.size, (0, 0, 0))
            bg.paste(raw, mask=raw.split()[3])
            img = bg
        else:
            img = raw.convert('RGB')
    return normalize_contrast(img)


def to_structural(img):
    arr = np.array(img.convert('L'))
    _, mask = cv2.threshold(arr, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    edges = cv2.Canny(arr, 50, 150)
    combined = cv2.bitwise_or(mask, edges)
    return Image.fromarray(combined).convert('RGB')


def letterbox_resize(img, size=IMG_SIZE, fill=0):
    w, h = img.size
    scale = size / max(w, h)
    new_w, new_h = max(1, round(w * scale)), max(1, round(h * scale))
    resized = img.resize((new_w, new_h), Image.BILINEAR)
    canvas = Image.new(img.mode, (size, size), fill)
    canvas.paste(resized, ((size - new_w) // 2, (size - new_h) // 2))
    return canvas


In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

infer_tf = transforms.Compose([
    transforms.Lambda(lambda img: letterbox_resize(img, IMG_SIZE)),
    transforms.Lambda(to_structural),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class AxisClassifier(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.backbone = timm.create_model(MODEL_NAME, pretrained=False, num_classes=0)
        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(self.backbone.num_features, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats)


def load_model(model_path):
    if not os.path.isfile(model_path):
        raise FileNotFoundError(f'Model file not found: {model_path}')
    model = AxisClassifier(num_classes=NUM_CLASSES).to(device)
    ckpt = torch.load(model_path, map_location=device)
    state_dict = ckpt.get('state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
    model.load_state_dict(state_dict)
    model.eval()
    return model


model = load_model(MODEL_PATH)


In [ ]:
class InferenceDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img = load_image(path)
        img = self.transform(img)
        return img, path


image_paths = sorted(
    os.path.join(IMAGE_DIR, f)
    for f in os.listdir(IMAGE_DIR)
    if not f.startswith('.') and is_image_file(f)
)
if len(image_paths) == 0:
    raise RuntimeError(f'No valid images found in {IMAGE_DIR}')

ds = InferenceDataset(image_paths, infer_tf)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


In [ ]:
pred_records = []

with torch.no_grad():
    for imgs, paths in dl:
        imgs = imgs.to(device)
        probs = torch.softmax(model(imgs), dim=1)
        confs, pred_idx = probs.max(dim=1)
        for i in range(len(paths)):
            pidx = int(pred_idx[i].item())
            pred_records.append({
                'path': paths[i],
                'filename': os.path.basename(paths[i]),
                'pred_class_idx': pidx,
                'pred_class': CLASSES[pidx],
                'confidence': float(confs[i].item()),
                'probs': probs[i].cpu().numpy(),
            })

pred_counts = Counter(r['pred_class'] for r in pred_records)
print(f'Inference complete for {len(pred_records)} images.')
for c in CLASSES:
    n = pred_counts.get(c, 0)
    pct = (n / len(pred_records)) * 100 if pred_records else 0.0
    print(f'{c:10s}: {n:5d} ({pct:6.2f}%)')

conf = np.array([r['confidence'] for r in pred_records], dtype=np.float32)
print(f'\nConfidence  mean={conf.mean():.4f}  median={np.median(conf):.4f}  min={conf.min():.4f}  max={conf.max():.4f}')


In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(x=CLASSES, y=[pred_counts.get(c, 0) for c in CLASSES], palette='viridis')
plt.title('Predicted Class Counts')
plt.xlabel('Predicted Class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
k = min(12, len(pred_records))
sample = sorted(pred_records, key=lambda x: x['confidence'])[:k]

if k > 0:
    cols = 4
    rows = (k + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.5 * rows))
    axes = np.array(axes).reshape(-1)
    for i, rec in enumerate(sample):
        img = load_image(rec['path'])
        axes[i].imshow(img)
        axes[i].set_title(f"{rec['filename']}\nPred: {rec['pred_class']} ({rec['confidence']:.2%})", fontsize=8)
        axes[i].axis('off')
    for j in range(k, len(axes)):
        axes[j].axis('off')
    plt.suptitle('Lowest-Confidence Predictions', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
